In [0]:
from pyspark.sql.functions import lit

catalog_name = "dp"

bronze_schema = "dp_1_bronze"
silver_schema = "dp_2_silver"
gold_schema = "dp_3_gold"

volume_sales = "sales"

source_path = "/Volumes/databricks_simulated_retail_customer_data/v02/subsidiary_daily_orders/bright_home_orders"
sink_path = f"/Volumes/{catalog_name}/{bronze_schema}"

date_on_file = "2025-11-05"
file_name = f"bsh_orders_{date_on_file}.csv"

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema}")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{bronze_schema}.{volume_sales}")

In [0]:

dbutils.fs.cp(f"{source_path}/{file_name}", f"{sink_path}/{volume_sales}/{file_name}")


In [0]:
## Check file
sql_result = (
    spark.sql(f"LIST '{sink_path}/{volume_sales}/'")
    .withColumn('volume', lit(f"{volume_sales}"))
)

display(sql_result)

In [0]:
sql_query_result = (
    spark.sql(f"""
        SELECT '{volume_sales}' as volume_name,
            COUNT(*) as total_rows,
            _metadata.file_name as file_name
        FROM read_files('{sink_path}/{volume_sales}')
        GROUP BY _metadata.file_name
    """)
)

display(sql_query_result)

In [0]:
%sql
SELECT source_file, count(1) FROM dp.dp_1_bronze.sales_bronze_raw
GROUP BY source_file

In [0]:
%sql
SELECT *
FROM dp.dp_1_bronze.sales_bronze_raw
WHERE customer_id IS NULL

In [0]:
%sql
SELECT event_type,
  details:flow_progress.data_quality.expectations
FROM event_log(TABLE(dp.dp_2_silver.sales_silver_dp))
WHERE event_type = 'flow_progress' 
AND details:flow_progress.data_quality.expectations IS NOT NULL

In [0]:
%sql
SELECT * FROM dp.dp_3_gold.sales_analytics